
# Customer Value Analytics: CLV Estimation, Revenue at Risk, and Retention Roadmap

Quantifying the business impact of churn through customer lifetime value proxies and revenue concentration analysis.

The notebook includes:

- Customer Lifetime Value (CLV) Estimation
- Consumption Segmentation (Quartiles)
- Revenue at Risk Analysis
- Customer Insights
- Recommendations
- Retention Strategy

In [2]:

# load modeling dataset

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("D:\Data Analysis\customer_churn_analysis\modeling_data.csv")


### Customer Lifetime Value (CLV) Proxy

CLV estimates the overall value of a customer to the business.

Higher CLV generally indicates:

* Higher customer value
* Greater revenue contribution
* Higher revenue risk if churn occurs
* Higher priority for retention efforts


In [22]:

#   Customer Lifetime Value (CLV) proxy
# Proxy: avg net_margin × avg tenure_years (simple, no discounting)

churned_df = df[df['churn'] == 1]
retained_df = df[df['churn'] == 0]

clv_retained = retained_df['tenure_years'].mean() * retained_df['net_margin'].mean()
clv_churned = churned_df['tenure_years'].mean() * churned_df['net_margin'].mean()

print('Customer Lifetime Value Proxy (avg net_margin × avg tenure)')
print('-' * 58)

print(
    f"Retained customers : avg tenure = {retained_df['tenure_years'].mean():.1f} yr, "
    f"avg margin = £{retained_df['net_margin'].mean():.2f}, "
    f"CLV ≈ £{clv_retained:.0f}"
)

print(
    f"Churned customers  : avg tenure = {churned_df['tenure_years'].mean():.1f} yr, "
    f"avg margin = £{churned_df['net_margin'].mean():.2f}, "
    f"CLV ≈ £{clv_churned:.0f}"
)

print()

tenure_gap = retained_df['tenure_years'].mean() - churned_df['tenure_years'].mean()
clv_loss = tenure_gap * churned_df['net_margin'].mean()

print('CLV lost per churned customer (tenure gap × churned margin):')
print(f'  Tenure gap : {tenure_gap:.2f} years')
print(f'  CLV loss   : £{clv_loss:.2f} per churned customer')
print(f'  Total CLV gap across all churned: £{clv_loss * len(churned_df):,.0f}')


Customer Lifetime Value Proxy (avg net_margin × avg tenure)
----------------------------------------------------------
Retained customers : avg tenure = 5.0 yr, avg margin = £185.06, CLV ≈ £919
Churned customers  : avg tenure = 4.6 yr, avg margin = £228.36, CLV ≈ £1041

CLV lost per churned customer (tenure gap × churned margin):
  Tenure gap : 0.41 years
  CLV loss   : £93.11 per churned customer
  Total CLV gap across all churned: £132,125


---
### Churn Rate by Consumption Quartile

Customers are grouped into four quartiles based on their consumption levels.

This helps identify:

* Differences in churn behavior across usage segments
* The relationship between consumption and churn
* Customer groups with higher churn risk
* Opportunities for targeted retention strategies

In [25]:

#   Churn rate by consumption quartile

df['cons_quartile'] = pd.qcut(df['cons_12m'], q=4, labels=['Q1 Low', 'Q2 Low-medium', 'Q3 Medium-high', 'Q4 High'])

cons_churn = (df.groupby('cons_quartile', observed=False).agg(
    churn_rate=('churn', 'mean'),
    customer_count=('churn', 'count'),
    total_net_margin=('net_margin', 'sum'),
    churned_customers=('churn', 'sum')).assign(churn_rate=lambda x: (x['churn_rate'] * 100).round(2))
)

print('Churn Rate by Consumption Quartile')
print(cons_churn.to_string())
print()

bounds = pd.qcut(df['cons_12m'], q=4, retbins=True)[1]

print('Quartile boundaries (kWh/yr):')
for i, (lo, hi) in enumerate(zip(bounds[:-1], bounds[1:]), start=1):          # prints quartile boundaries for the values in cons_12m column
    print(f'  Q{i}: {lo:>10,.0f} – {hi:>10,.0f}')


Churn Rate by Consumption Quartile
                churn_rate  customer_count  total_net_margin  churned_customers
cons_quartile                                                                  
Q1 Low                9.34            3652         158385.01                341
Q2 Low-medium         9.83            3651         387764.29                359
Q3 Medium-high        9.97            3651         828624.66                364
Q4 High               9.72            3652        1389623.65                355

Quartile boundaries (kWh/yr):
  Q1:          0 –      5,675
  Q2:      5,675 –     14,116
  Q3:     14,116 –     40,764
  Q4:     40,764 –  6,207,104


---
### Churn Rate by Net Margin Quartile

Customers are grouped into four quartiles based on their net margin contribution.

This helps to identify:

* Differences in churn behavior across profitability segments
* The relationship between customer profitability and churn
* High-margin customer groups with elevated churn risk
* Opportunities to prioritize retention efforts for valuable customers

In [27]:

#    Churn rate by net margin quartile

df['margin_quartile'] = pd.qcut(df['net_margin'], q=4, labels=['Q1 Low', 'Q2 Low-medium', 'Q3 Medium-high', 'Q4 High'])

margin_churn = (df.groupby('margin_quartile', observed=False).agg(
          churn_rate=('churn', 'mean'),
          customer_count=('churn', 'count'),
          total_margin=('net_margin', 'sum'),
          churned_count=('churn', 'sum')).assign(churn_rate=lambda x: (x['churn_rate'] * 100).round(2))
)

print('Churn Rate by Net Margin Quartile')
print(margin_churn.to_string())
print()

q4_churn = margin_churn.loc['Q4 High', 'churn_rate']
q1_churn = margin_churn.loc['Q1 Low', 'churn_rate']

print(f'⚠️ Highest-margin customers (Q4) churn at {q4_churn}% '
    f'vs {q1_churn}% for Q1 — PowerCo is losing its best accounts.')


Churn Rate by Net Margin Quartile
                 churn_rate  customer_count  total_margin  churned_count
margin_quartile                                                         
Q1 Low                 9.36            3652      96902.74            342
Q2 Low-medium          9.15            3651     288909.59            334
Q3 Medium-high         9.45            3651     604672.91            345
Q4 High               10.90            3652    1773912.37            398

⚠️ Highest-margin customers (Q4) churn at 10.9% vs 9.36% for Q1 — PowerCo is losing its best accounts.


---
### Revenue at Risk Analysis

This breakdown quantifies the financial impact of customer churn and estimates potential revenue retention through successful intervention strategies.

Key metrics include:

* Total and churned customer counts
* Overall churn and retention rates
* Total net margin lost due to churn
* Average margin of churned customers
* Average revenue per customer (ARPU)
* Potential revenue retained at different customer save rates


In [36]:

#    Revenue at risk — full breakdown 

total_customers = len(df)
churned_customers = df['churn'].sum()
retention_rate = (1 - df['churn'].mean()) * 100

total_margin_lost = churned_df['net_margin'].sum()
avg_margin_churned = churned_df['net_margin'].mean()
arpu = df['net_margin'].mean()

save_rates = [0.10, 0.20, 0.30, 0.40, 0.50]

print('Revenue at Risk — Summary')
print('-' * 40)

print(f'Total customers       : {total_customers:>8,}')
print(f'Churned customers     : {churned_customers:>8,}')
print(f'Churn rate            : {df["churn"].mean()*100:>7.1f}%')
print(f'Retention rate        : {retention_rate:>7.1f}%')
print(f'Total net margin lost : £{total_margin_lost:>10,.2f}')
print(f'Avg margin (churned)  : £{avg_margin_churned:>10.2f}')
print(f'ARPU (all customers)  : £{arpu:>10.2f}')

print()
print('Revenue Retained at Different Intervention Save Rates:')
print('-' * 60)

for sr in save_rates:
    retained_rev = total_margin_lost * sr
    saved_cust = int(churned_customers * sr)

    print(
        f'  {int(sr*100)}% save rate → '
        f'{saved_cust:>4} customers saved → '
        f'£{retained_rev:>10,.0f} retained'
    )


Revenue at Risk — Summary
----------------------------------------
Total customers       :   14,606
Churned customers     :    1,419
Churn rate            :     9.7%
Retention rate        :    90.3%
Total net margin lost : £324,045.59
Avg margin (churned)  : £    228.36
ARPU (all customers)  : £    189.26

Revenue Retained at Different Intervention Save Rates:
------------------------------------------------------------
  10% save rate →  141 customers saved → £    32,405 retained
  20% save rate →  283 customers saved → £    64,809 retained
  30% save rate →  425 customers saved → £    97,214 retained
  40% save rate →  567 customers saved → £   129,618 retained
  50% save rate →  709 customers saved → £   162,023 retained


---
### High-Value At-Risk Customer Segment

This focuses on customers who generate high value for the business and are at greater risk of churning.

This helps ro identify:

* The size of the high-value at-risk segment
* Churn levels within this customer group
* Total margin potentially at risk

In [38]:

# High-value at-risk segment identification

margin_p75 = df['net_margin'].quantile(0.75)          # calculates 75th percentile (third quartile, Q3) of net_margin column
                                                     # Only 25% of the net margins are above that value  
high_value = df['net_margin'] >= margin_p75
early_tenure = df['tenure_years'] < 3
near_expiry = df['months_to_end'] < 6

at_risk = early_tenure | near_expiry

priority_segment = df[high_value & at_risk]

print('Priority Intervention Segment: High-Value & At-Risk')
print('-' * 55)

print('Definition : net_margin ≥ P75 AND (tenure < 3 years OR contract ending < 6 months)')

print(
    f'Segment size: {len(priority_segment):,} customers '
    f'({len(priority_segment)/total_customers*100:.1f}% of base)'
)

print(f'Churn rate : {priority_segment["churn"].mean()*100:.1f}%')

margin_at_risk = priority_segment.loc[priority_segment['churn'] == 1, 'net_margin'].sum()

print(f'Total margin at risk in segment: £{margin_at_risk:,.0f}')

print()
print('Segment profile:')

print(priority_segment.groupby('churn')[['net_margin', 'tenure_years', 'cons_12m']].mean().round(2))


Priority Intervention Segment: High-Value & At-Risk
-------------------------------------------------------
Definition : net_margin ≥ P75 AND (tenure < 3 years OR contract ending < 6 months)
Segment size: 1,838 customers (12.6% of base)
Churn rate : 10.7%
Total margin at risk in segment: £110,761

Segment profile:
       net_margin  tenure_years   cons_12m
churn                                     
0          484.68          4.87  401915.62
1          562.24          4.17  158220.13


---
### KPI Summary

This table summarizes the key findings from the churn analysis into a single business performance overview.

Key metrics include:

* Customer churn and retention rates
* Customer base and churn volume
* Revenue and net margin impact of churn
* Customer value indicators (ARPU and CLV loss)
* Churn trends across tenure segments
* Churn comparison by product ownership

In [41]:

# Final KPI summary table 

kpis = {
    'Churn Rate': f"{df['churn'].mean()*100:.1f}%",
    'Retention Rate': f"{(1-df['churn'].mean())*100:.1f}%",
    'Total Customers': f"{total_customers:,}",
    'Churned Customers': f"{churned_customers:,}",
    'Net Margin Lost (annual)': f"£{total_margin_lost:,.0f}",
    'Avg Net Margin — Churned': f"£{avg_margin_churned:.2f}",
    'Avg Net Margin — Retained': f"£{retained_df['net_margin'].mean():.2f}",
    'ARPU (all customers)': f"£{arpu:.2f}",
    'CLV Loss per Churned Customer': f"£{clv_loss:.2f}",
    'Early-Tenure Churn Rate': f"{df[df['tenure_years'] < 3]['churn'].mean()*100:.1f}%",
    'Long-Tenure Churn Rate (10+y)': f"{df[df['tenure_years'] >= 10]['churn'].mean()*100:.1f}%",
    'Gas Bundle Churn Rate': f"{df[df['has_gas'] == 't']['churn'].mean()*100:.1f}%",
    'Elec-Only Churn Rate': f"{df[df['has_gas'] == 'f']['churn'].mean()*100:.1f}%",
}

print('PowerCo Churn Analysis — Final KPI Summary')
print('-' * 50)

for k, v in kpis.items():
    print(f'{k:<38} {v:>10}')


PowerCo Churn Analysis — Final KPI Summary
--------------------------------------------------
Churn Rate                                   9.7%
Retention Rate                              90.3%
Total Customers                            14,606
Churned Customers                           1,419
Net Margin Lost (annual)                 £324,046
Avg Net Margin — Churned                  £228.36
Avg Net Margin — Retained                 £185.06
ARPU (all customers)                      £189.26
CLV Loss per Churned Customer              £93.11
Early-Tenure Churn Rate                     15.4%
Long-Tenure Churn Rate (10+y)                7.1%
Gas Bundle Churn Rate                        8.2%
Elec-Only Churn Rate                        10.1%


---
## Findings & Recommendations
#### 1. Price Is NOT the Primary Churn Driver
- Off-peak price difference between churned and retained customers is <0.05% statistically insignificant (p=0.44). Price changes over 2015 show no meaningful association with churn. The price-sensitivity hypothesis is not supported by the data.

#### 2. Early-Tenure Customers Are the Highest-Risk Segment
- Customers in their 1st–3rd year churn at 15.3% more than double the rate of 5–10 year customers (7.3%). The first 3 years of the customer relationship are the single most critical retention window PowerCo must invest in.

#### 3. PowerCo Is Losing Its Best Customers
- Churned customers have 23% higher average net margin (£228 vs £185, p<0.0001) and cluster in the highest-margin quartile (10.9% churn rate). Competitors are targeting PowerCo's most profitable SMEs, this is the most urgent commercial risk.

#### 4. Low Consumption = Low Stickiness
- Churned customers consume 53% less electricity annually than retained customers (78,862 vs 167,867 kWh, p<0.0001). Low-consumption SMEs are less embedded in PowerCo's infrastructure and face lower switching costs.

#### 5. Multi-Product Bundling Is a Protective Moat
- Customers with a gas add-on churn at 8.2% vs 10.1% for electricity-only customers (χ²=8.43, p=0.004). Gas bundlers represent a 19% lower churn risk and a cross-sell opportunity for the 82% of customers who are electricity-only.

#### 6. Contract Proximity Is a Known Vulnerability
- Contract end date proximity is the 4th most important predictor in the Random Forest model (8.9% importance). Competitors clearly time acquisition outreach around contract renewals, PowerCo must beat them to the conversation.

#### 7. Sales Channel Quality Varies Significantly
- Channel A acquires 46% of the customer base but drives a 12.1% churn rate vs 5.6% for Channel C. Poor onboarding quality or customer-fit mismatch in Channel A is likely, a channel audit should be an immediate action item.

#### 8. £324K Net Margin Is Lost Annually — Immediate Action Required
- The 1,419 churned customers represent £324,046 in net margin. A targeted retention programme with a modest 30% save rate on high-risk customers would retain approximately £97K of this margin annually, at a fraction of customer acquisition cost.

---
## Retention Strategy
### A Three-Phase Roadmap for Customer Retention

### 🔍 0–30 Days · Immediate
> **Stop the Bleeding**

- Flag all customers with tenure <3 yr + high margin — assign dedicated account managers
- Identify all contracts expiring within 90 days — trigger proactive renewal outreach
- Deploy retention offers to Channel A customers with above-median consumption
- Audit Channel A onboarding process for quality gaps vs Channel C
- Generate churn probability scores for all 14,606 customers using current model

### 🛠️ 1–6 Months · Short-Term
> **Structural Fixes**

- Launch gas add-on bundle campaign targeting electricity-only SMEs in yr 1–3
- Redesign contract renewal process — offer incentives at 6 and 3 months prior to expiry
- Create early-tenure onboarding program (months 0–18): product education, usage reviews
- Build consumption monitoring alerts — flag accounts with declining usage trend
- Test personalised pricing/discounts for highest-risk, highest-value segments only
- Enhance churn model with XGBoost + SMOTE; target ROC-AUC >0.75

### 🌱 6–24 Months · Long-Term
> **Sustainable Retention**

- Build a real-time churn scoring pipeline — weekly model refresh per customer
- Launch SME loyalty programme with tiered benefits by tenure milestone
- Invest in customer success team focused on multi-product adoption
- Develop product modification engagement cadence — proactively offer plan reviews
- Commission customer satisfaction surveys — fill behavioral data gap
- Integrate competitor pricing intelligence to identify externally at-risk accounts

---
## Conclusion
This analysis found that customer churn at PowerCo is driven more by customer behaviour and engagement than by pricing. The highest-risk groups are early-tenure customers, low-consumption accounts, and customers acquired through certain sales channels. Particularly, PowerCo is losing many of its most profitable customers, resulting in an estimated £324K in annual net margin loss.

The findings suggest that focusing on early customer retention, proactive contract renewals, gas bundling, and targeted retention campaigns can significantly reduce churn. Overall, this project demonstrates how data-driven insights can help PowerCo improve customer retention, protect revenue, and support more effective business decisions.
